# African Proverb Few-Shot Evaluation Monitor
This notebook helps monitor and analyze few-shot evaluation runs

In [ ]:
import sys
sys.path.append('./src')

import json
import pandas as pd
from pathlib import Path
from proverb.engine.args import _parse_args
from proverb.data.loader import load_proverb_dataset
from proverb.model.loader import load_tokenizer

## Load Configuration

In [ ]:
# Set parameters
location = "Kenya"
language = "gikuyu"
task_type = "gen_eng_literal"
few_shot_num = 3

print(f"Location: {location}")
print(f"Language: {language}")
print(f"Task: {task_type}")
print(f"Few-shot examples: {few_shot_num}")

## Load Dataset and Inspect Few-Shot Examples

In [ ]:
from proverb.data.loader import ProverbDataset
from proverb.data.chat_template import get_template_and_fix_tokenizer
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
template = get_template_and_fix_tokenizer("qwen3", tokenizer)

# Load dataset
dataset = ProverbDataset(
    dataset_dir="dataset/African-Proverbs/Data",
    location=location,
    language=language,
    task_type=task_type,
    few_shot_num=few_shot_num,
)

print(f"Dataset size: {len(dataset)}")

## Inspect Sample with Few-Shot Prompt

In [ ]:
# Get first sample
sample = dataset[0]

print("=" * 80)
print("SAMPLE INPUT:")
print("=" * 80)
print(sample['input'])
print("\n" + "=" * 80)
print("EXPECTED OUTPUT:")
print("=" * 80)
print(sample['output'])
print("\n" + "=" * 80)
print("FULL MESSAGES:")
print("=" * 80)
for msg in sample['messages']:
    print(f"\n[{msg['role'].upper()}]")
    print(msg['content'])

## View Multiple Samples

In [ ]:
# View first N samples
n_samples = 3

for i in range(min(n_samples, len(dataset))):
    sample = dataset[i]
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    print(f"Proverb: {sample['input']}")
    print(f"Expected: {sample['output']}")

## Monitor Evaluation Results

In [ ]:
# Load evaluation results
output_dir = Path("outputs/qwen3-4b")

results = []
for result_file in output_dir.rglob("evaluation_results.json"):
    with open(result_file) as f:
        data = json.load(f)
        results.extend(data)

if results:
    df = pd.json_normalize(results)
    print(df.to_string())
else:
    print("No results found yet")

## View Generated Predictions

In [ ]:
# Load predictions
pred_file = output_dir / f"qwen3-4b-{task_type}-{location}" / f"generated_predictions_{location}_{language}.jsonl"

if pred_file.exists():
    predictions = []
    with open(pred_file) as f:
        for line in f:
            predictions.append(json.loads(line))
    
    pred_df = pd.DataFrame(predictions)
    print(f"Total predictions: {len(pred_df)}")
    print(pred_df.head(10))
else:
    print(f"Prediction file not found: {pred_file}")

## Compare Predictions vs Ground Truth

In [ ]:
if pred_file.exists():
    for i, pred in enumerate(predictions[:5]):
        print(f"\n{'='*80}")
        print(f"Example {i+1}")
        print(f"{'='*80}")
        print(f"Input: {pred.get('input', 'N/A')}")
        print(f"\nPredicted: {pred.get('predict', 'N/A')}")
        print(f"\nGround Truth: {pred.get('label', 'N/A')}")